# vary-n-voxels

In [ ]:
from pathlib import Path

import numpy as np
import seaborn as sns
import xarray as xr
from bonner.plotting import save_figure
from matplotlib import pyplot as plt
from tqdm.auto import tqdm

from lib.datasets import (
    compute_shared_stimuli,
    filter_by_stimulus,
    nsd,
    sample_neuroids,
    split_by_repetition,
)
from lib.spectra import (
    compute_within_individual_spectra,
    plot_spectra,
)
from lib.utilities import JOURNAL_MATPLOTLIBRC

FIGURES_HOME = Path.cwd().parent / "figures"
FIGURES_HOME.mkdir(exist_ok=True, parents=True)

sns.set_theme(context="paper", style="ticks", rc=JOURNAL_MATPLOTLIBRC)

REFERENCE_SUBJECT = 0

## load datasets

In [ ]:
n_voxels = np.geomspace(100, 10**4, num=5).astype(int)

dataset = nsd.load_dataset(
    subject=REFERENCE_SUBJECT,
    preprocessing="fithrf",
    roi="general",
    z_score=True,
)
repeated_stimuli = compute_shared_stimuli([dataset], n_repetitions=2)

datasets = {
    n_voxels_: sample_neuroids(dataset, n_neuroids=n_voxels_, random_state=0)
    for n_voxels_ in n_voxels
}

datasets = {
    n_voxels_: split_by_repetition(
        filter_by_stimulus(dataset, stimuli=repeated_stimuli),
        n_repetitions=2,
    )
    for n_voxels_, dataset in datasets.items()
}

## compute spectra

In [ ]:
spectra = {
    normalize: xr.concat(
        [
            compute_within_individual_spectra(
                {0: datasets_},
                normalize=normalize,
                n_permutations=5_000,
                n_bootstraps=0,
            ).expand_dims(n_voxels=[str(n_voxels_)])
            for n_voxels_, datasets_ in tqdm(
                datasets.items(),
                desc="n_voxels",
                leave=False,
            )
        ],
        dim="n_voxels",
    )
    for normalize in tqdm((True, False), desc="normalize", leave=False)
}

## plot spectra

In [ ]:
fig, axes = plt.subplots(
    figsize=(5, 3),
    ncols=2,
    sharex=True,
    sharey=False,
)

for normalize, ax in zip((False, True), axes.flat, strict=False):
    plot_spectra(
        spectra=spectra[normalize],
        ax=ax,
        hue="n_voxels",
        hue_order=[str(_) for _ in reversed(n_voxels)],
        hue_labels=[str(_) for _ in reversed(n_voxels)],
        palette="viridis",
        marker="s",
        hide_insignificant=True,
        null_quantile=0.999,
    )
    ax.set_xscale("log")
    ax.set_yscale("log")

    ax.set_xlim(left=1, right=1e4)
    ax.set_xticks([1, 1e1, 1e2, 1e3, 1e4])
    if normalize:
        ax.set_ylim(bottom=1e-8, top=1e-1)
        ax.set_yticks([10**x for x in range(-8, 0)])
    else:
        ax.set_ylim(bottom=1e-4, top=1e3)
        ax.set_yticks([10**x for x in range(-4, 4)])
    prefix = "after" if normalize else "before"
    ax.set_title(f"{prefix} normalization")

axes[1].legend(
    loc="lower left",
    title="# voxels",
    alignment="right",
    reverse=True,
)
axes[0].set_ylabel("covariance")
fig.supxlabel("rank", x=0.53, y=0.05)

fig.suptitle(f"within-subject, subject {1 + REFERENCE_SUBJECT}", y=0.95)
fig.tight_layout()

save_figure(
    fig,
    filepath=FIGURES_HOME / "vary-n-voxels.pdf",
)